In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path 

In [2]:
## project paths 
PROJECT_ROOT  = Path.cwd().parent
RAW_DIR  = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project Root:", PROJECT_ROOT)
print("Raw Data:", RAW_DIR)
print("Processed Data:", PROCESSED_DIR)

Project Root: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone
Raw Data: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\data\raw
Processed Data: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\data\processed


In [3]:
## load NAV history 
nav_df  = pd.read_csv(RAW_DIR / "02_nav_history.csv")

print("shape" ,nav_df.shape)

print("columns:" , nav_df.columns.to_list)
print("\nFirst 5 Rows:")
display(nav_df.head())

print("mising values ")
print(nav_df.isnull().sum())
print("\nDuplicate Rows:")
print(nav_df.duplicated().sum())


shape (46000, 3)
columns: <bound method IndexOpsMixin.tolist of Index(['amfi_code', 'date', 'nav'], dtype='object')>

First 5 Rows:


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


mising values 
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate Rows:
0


In [4]:
# Create a copy of original data
clean_nav = nav_df.copy()

# 1. Convert date column to datetime
clean_nav["date"] = pd.to_datetime(clean_nav["date"])

# 2. Convert NAV to numeric
clean_nav["nav"] = pd.to_numeric(clean_nav["nav"], errors="coerce")

# 3. Sort by AMFI code and date
clean_nav = clean_nav.sort_values(
    by=["amfi_code", "date"]
)

# 4. Forward-fill missing NAV within each scheme
clean_nav["nav"] = (
    clean_nav.groupby("amfi_code")["nav"].ffill()
)

# 5. Remove duplicate records
clean_nav = clean_nav.drop_duplicates(
    subset=["amfi_code", "date"]
)

# 6. Validate NAV > 0
clean_nav = clean_nav[clean_nav["nav"] > 0]

# 7. Save cleaned dataset
clean_nav.to_csv(
    PROCESSED_DIR / "clean_nav.csv",
    index=False
)

print("Original Shape:", nav_df.shape)
print("Cleaned Shape:", clean_nav.shape)
print("Missing Values:\n", clean_nav.isnull().sum())
print("Duplicate Rows:", clean_nav.duplicated().sum())
print("\nNAV Cleaning Completed!")

Original Shape: (46000, 3)
Cleaned Shape: (46000, 3)
Missing Values:
 amfi_code    0
date         0
nav          0
dtype: int64
Duplicate Rows: 0

NAV Cleaning Completed!


In [5]:
## investor transsactions 
transactions_df  = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
print("Shape:", transactions_df.shape)
print("\nColumns:")
print(transactions_df.columns.tolist())
print("\nData Types:")
print(transactions_df.dtypes)
print("\nFirst 5 Rows:")
display(transactions_df.head())
print("\nMissing Values:")
print(transactions_df.isnull().sum())
print("\nDuplicate Rows:")
print(transactions_df.duplicated().sum())

Shape: (32778, 13)

Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Data Types:
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object

First 5 Rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending



Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Duplicate Rows:
0


In [6]:
# Create a copy of original dataset
clean_transactions = transactions_df.copy()

# 1. Standardize transaction type
clean_transactions["transaction_type"] = (
    clean_transactions["transaction_type"]
    .str.strip()
    .str.lower()
    .replace({
        "lump sum": "Lumpsum",
        "lumpsum": "Lumpsum",
        "sip": "SIP",
        "redemption": "Redemption"
    })
)

# 2. Convert transaction date to datetime
clean_transactions["transaction_date"] = pd.to_datetime(
    clean_transactions["transaction_date"],
    errors="coerce"
)

# 3. Convert amount to numeric
clean_transactions["amount_inr"] = pd.to_numeric(
    clean_transactions["amount_inr"],
    errors="coerce"
)

# 4. Validate transaction amount > 0
clean_transactions = clean_transactions[
    clean_transactions["amount_inr"] > 0
]

# 5. Check KYC status values
print("KYC Status Values:")
print(clean_transactions["kyc_status"].unique())

# 6. Check transaction types
print("\nTransaction Types:")
print(clean_transactions["transaction_type"].value_counts())

# 7. Remove duplicate transactions
clean_transactions = clean_transactions.drop_duplicates()

# 8. Save cleaned dataset
clean_transactions.to_csv(
    PROCESSED_DIR / "clean_transactions.csv",
    index=False
)

print("\nOriginal Shape:", transactions_df.shape)
print("Cleaned Shape:", clean_transactions.shape)
print("\nMissing Values:")
print(clean_transactions.isnull().sum())

print("\nTransaction Cleaning Completed!")

KYC Status Values:
['Verified' 'Pending']

Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Original Shape: (32778, 13)
Cleaned Shape: (32778, 13)

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Transaction Cleaning Completed!


In [7]:
clean_transactions.head(10)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending
5,INV001497,2024-01-01,101208,SIP,3295,Maharashtra,Mumbai,T30,36-45,Male,56.8,Mandate,Verified
6,INV000786,2024-01-01,101208,SIP,15047,Madhya Pradesh,Bhopal,B30,26-35,Male,17.9,Mandate,Verified
7,INV000616,2024-01-01,100016,SIP,933,Maharashtra,Nashik,B30,36-45,Male,50.4,Mandate,Verified
8,INV003670,2024-01-01,119120,SIP,10672,Punjab,Chandigarh,T30,36-45,Male,38.4,Net Banking,Verified
9,INV002054,2024-01-01,148567,SIP,1111,Telangana,Hyderabad,T30,26-35,Male,17.8,Net Banking,Verified


In [8]:
performance_df = pd.read_csv(
    RAW_DIR / "07_scheme_performance.csv"
)

print("Shape:", performance_df.shape)
print("\nColumns:")
print(performance_df.columns.tolist())

print("\nData Types:")
print(performance_df.dtypes)

print("\nFirst 5 Rows:")
display(performance_df.head())

print("\nMissing Values:")
print(performance_df.isnull().sum())

print("\nDuplicate Rows:")
print(performance_df.duplicated().sum())

Shape: (40, 19)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Data Types:
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

First 5 Rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low



Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

Duplicate Rows:
0


In [9]:
# Create a copy of original dataset
clean_performance = performance_df.copy()

# 1. Convert return-related columns to numeric
numeric_columns = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct"
]

for col in numeric_columns:
    clean_performance[col] = pd.to_numeric(
        clean_performance[col], errors="coerce"
    )

# 2. Flag negative Sharpe ratios
clean_performance["negative_sharpe_flag"] = (
    clean_performance["sharpe_ratio"] < 0
)

# 3. Validate expense ratio (0.1% to 2.5%)
clean_performance["expense_ratio_valid"] = (
    clean_performance["expense_ratio_pct"].between(0.1, 2.5)
)

# 4. Remove duplicate rows
clean_performance = clean_performance.drop_duplicates()

# 5. Save cleaned dataset
clean_performance.to_csv(
    PROCESSED_DIR / "clean_performance.csv",
    index=False
)

# 6. Display cleaning report
print("Original Shape:", performance_df.shape)
print("Cleaned Shape:", clean_performance.shape)

print("\nNegative Sharpe Ratios:")
print(clean_performance["negative_sharpe_flag"].sum())

print("\nInvalid Expense Ratios:")
print((~clean_performance["expense_ratio_valid"]).sum())

print("\nMissing Values:")
print(clean_performance.isnull().sum())

print("\nScheme Performance Cleaning Completed!")

Original Shape: (40, 19)
Cleaned Shape: (40, 21)

Negative Sharpe Ratios:
0

Invalid Expense Ratios:
0

Missing Values:
amfi_code               0
scheme_name             0
fund_house              0
category                0
plan                    0
return_1yr_pct          0
return_3yr_pct          0
return_5yr_pct          0
benchmark_3yr_pct       0
alpha                   0
beta                    0
sharpe_ratio            0
sortino_ratio           0
std_dev_ann_pct         0
max_drawdown_pct        0
aum_crore               0
expense_ratio_pct       0
morningstar_rating      0
risk_grade              0
negative_sharpe_flag    0
expense_ratio_valid     0
dtype: int64

Scheme Performance Cleaning Completed!


In [10]:
## sqlite database 

import sqlite3 
from sqlalchemy import create_engine , inspect
from pathlib import Path 

In [11]:
# Define database directory
DB_DIR = PROJECT_ROOT / "data" / "db"
DB_DIR.mkdir(parents=True, exist_ok=True)

# Database path
DB_PATH = DB_DIR / "bluestock_mf.db"

# Read schema.sql
schema_path = PROJECT_ROOT / "sql" / "schema.sql"

with open(schema_path, "r", encoding="utf-8") as file:
    schema_sql = file.read()

# Create database and execute schema
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(schema_sql)

# Create SQLAlchemy engine
engine = create_engine(f"sqlite:///{DB_PATH}")

print("Database created successfully!")
print("Database Path:", DB_PATH)

# Verify tables
inspector = inspect(engine)

print("\nTables Created:")
print(inspector.get_table_names())

Database created successfully!
Database Path: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\data\db\bluestock_mf.db

Tables Created:
['dim_date', 'dim_fund', 'fact_aum', 'fact_nav', 'fact_performance', 'fact_transactions']


In [12]:
# Load Fund Master Dataset
fund_df = pd.read_csv(
    RAW_DIR / "01_fund_master.csv"
)

# Inspect dataset
print("Dataset Shape:", fund_df.shape)

print("\nColumn Names:")
print(fund_df.columns.tolist())

print("\nFirst 5 Records:")
display(fund_df.head())

Dataset Shape: (40, 15)

Column Names:
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

First 5 Records:


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [13]:
# Load original fund master dataset
fund_df = pd.read_csv(
    RAW_DIR / "01_fund_master.csv"
)

# Create a copy
clean_fund = fund_df.copy()

# Remove extra spaces from text columns
text_columns = clean_fund.select_dtypes(
    include="object"
).columns

for col in text_columns:
    clean_fund[col] = clean_fund[col].str.strip()

# Remove duplicate AMFI codes
clean_fund = clean_fund.drop_duplicates(
    subset=["amfi_code"]
)

# Remove records without essential identifiers
clean_fund = clean_fund.dropna(
    subset=["amfi_code", "scheme_name"]
)

# Save cleaned dataset
clean_fund.to_csv(
    PROCESSED_DIR / "clean_fund_master.csv",
    index=False
)

print("Original Shape:", fund_df.shape)
print("Cleaned Shape:", clean_fund.shape)
print("\nMissing Values:")
print(clean_fund.isnull().sum())

print("\nFund Master Cleaning Completed!")

Original Shape: (40, 15)
Cleaned Shape: (40, 15)

Missing Values:
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

Fund Master Cleaning Completed!


In [14]:
# Load AUM dataset
aum_df = pd.read_csv(
    RAW_DIR / "03_aum_by_fund_house.csv"
)

print("Shape:", aum_df.shape)

print("\nColumns:")
print(aum_df.columns.tolist())

print("\nData Types:")
print(aum_df.dtypes)

print("\nFirst 5 Rows:")
display(aum_df.head())

print("\nMissing Values:")
print(aum_df.isnull().sum())

print("\nDuplicate Rows:")
print(aum_df.duplicated().sum())

Shape: (90, 5)

Columns:
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']

Data Types:
date               object
fund_house         object
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object

First 5 Rows:


,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168



Missing Values:
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

Duplicate Rows:
0


In [15]:
# Create a copy of original dataset
clean_aum = aum_df.copy()

# 1. Convert date to datetime
clean_aum["date"] = pd.to_datetime(
    clean_aum["date"],
    errors="coerce"
)

# 2. Convert numeric columns
numeric_columns = [
    "aum_lakh_crore",
    "aum_crore",
    "num_schemes"
]

for col in numeric_columns:
    clean_aum[col] = pd.to_numeric(
        clean_aum[col],
        errors="coerce"
    )

# 3. Validate positive AUM and number of schemes
clean_aum = clean_aum[
    (clean_aum["aum_crore"] > 0) &
    (clean_aum["num_schemes"] > 0)
]

# 4. Remove duplicate records
clean_aum = clean_aum.drop_duplicates()

# 5. Save cleaned dataset
clean_aum.to_csv(
    PROCESSED_DIR / "clean_aum.csv",
    index=False
)

# 6. Display cleaning report
print("Original Shape:", aum_df.shape)
print("Cleaned Shape:", clean_aum.shape)

print("\nMissing Values:")
print(clean_aum.isnull().sum())

print("\nDuplicate Rows:", clean_aum.duplicated().sum())

print("\nAUM Cleaning Completed!")

Original Shape: (90, 5)
Cleaned Shape: (90, 5)

Missing Values:
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

Duplicate Rows: 0

AUM Cleaning Completed!


In [16]:
# Load Monthly SIP Inflows dataset
sip_df = pd.read_csv(
    RAW_DIR / "04_monthly_sip_inflows.csv"
)

print("Shape:", sip_df.shape)

print("\nColumns:")
print(sip_df.columns.tolist())

print("\nData Types:")
print(sip_df.dtypes)

print("\nFirst 5 Rows:")
display(sip_df.head())

print("\nMissing Values:")
print(sip_df.isnull().sum())

print("\nDuplicate Rows:")
print(sip_df.duplicated().sum())

Shape: (48, 6)

Columns:
['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']

Data Types:
month                         object
sip_inflow_crore               int64
active_sip_accounts_crore    float64
new_sip_accounts_lakh        float64
sip_aum_lakh_crore           float64
yoy_growth_pct               float64
dtype: object

First 5 Rows:


,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01,11517,4.91,9.10,4.80,NaN
1,2022-02,11438,4.93,8.20,4.85,NaN
2,2022-03,12328,5.09,10.50,5.01,NaN
3,2022-04,11863,5.48,9.52,5.12,NaN
4,2022-05,12286,5.55,8.10,5.15,NaN



Missing Values:
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

Duplicate Rows:
0


In [17]:
print("First 15 rows:")
display(sip_df.head(15))

print("\nDuplicate Rows:", sip_df.duplicated().sum())

print("\nMissing YoY Growth Rows:")
display(sip_df[sip_df["yoy_growth_pct"].isnull()])

First 15 rows:


,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01,11517,4.91,9.10,4.80,NaN
1,2022-02,11438,4.93,8.20,4.85,NaN
2,2022-03,12328,5.09,10.50,5.01,NaN
3,2022-04,11863,5.48,9.52,5.12,NaN
4,2022-05,12286,5.55,8.10,5.15,NaN
5,2022-06,12276,5.60,7.50,5.18,NaN
6,2022-07,12140,5.65,9.21,5.25,NaN
7,2022-08,12694,5.71,8.90,5.31,NaN
8,2022-09,12976,5.80,9.40,5.41,NaN
9,2022-10,13040,5.93,9.52,5.55,NaN



Duplicate Rows: 0

Missing YoY Growth Rows:


,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01,11517,4.91,9.10,4.80,NaN
1,2022-02,11438,4.93,8.20,4.85,NaN
2,2022-03,12328,5.09,10.50,5.01,NaN
3,2022-04,11863,5.48,9.52,5.12,NaN
4,2022-05,12286,5.55,8.10,5.15,NaN
5,2022-06,12276,5.60,7.50,5.18,NaN
6,2022-07,12140,5.65,9.21,5.25,NaN
7,2022-08,12694,5.71,8.90,5.31,NaN
8,2022-09,12976,5.80,9.40,5.41,NaN
9,2022-10,13040,5.93,9.52,5.55,NaN


In [18]:
# Create a copy
clean_sip = sip_df.copy()

# Convert month into datetime format
clean_sip["month"] = pd.to_datetime(
    clean_sip["month"], format="%Y-%m", errors="coerce"
)

# Convert numeric columns
numeric_columns = [
    "sip_inflow_crore",
    "active_sip_accounts_crore",
    "new_sip_accounts_lakh",
    "sip_aum_lakh_crore",
    "yoy_growth_pct"
]

for col in numeric_columns:
    clean_sip[col] = pd.to_numeric(
        clean_sip[col], errors="coerce"
    )

# Remove duplicate records
clean_sip = clean_sip.drop_duplicates()

# Remove rows with invalid month
clean_sip = clean_sip.dropna(subset=["month"])

# Save cleaned dataset
clean_sip.to_csv(
    PROCESSED_DIR / "clean_monthly_sip.csv",
    index=False
)

# Verify cleaning
print("Original Shape:", sip_df.shape)
print("Cleaned Shape:", clean_sip.shape)

print("\nMissing Values:")
print(clean_sip.isnull().sum())

print("\nDuplicate Rows:", clean_sip.duplicated().sum())

print("\nMonthly SIP Cleaning Completed!")

Original Shape: (48, 6)
Cleaned Shape: (48, 6)

Missing Values:
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

Duplicate Rows: 0

Monthly SIP Cleaning Completed!


In [19]:
# Load Category Inflows dataset
category_df = pd.read_csv(
    RAW_DIR / "05_category_inflows.csv"
)

print("Shape:", category_df.shape)

print("\nColumns:")
print(category_df.columns.tolist())

print("\nData Types:")
print(category_df.dtypes)

print("\nFirst 5 Rows:")
display(category_df.head())

print("\nMissing Values:")
print(category_df.isnull().sum())

print("\nDuplicate Rows:")
print(category_df.duplicated().sum())

Shape: (144, 3)

Columns:
['month', 'category', 'net_inflow_crore']

Data Types:
month                object
category             object
net_inflow_crore    float64
dtype: object

First 5 Rows:


,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0



Missing Values:
month               0
category            0
net_inflow_crore    0
dtype: int64

Duplicate Rows:
0


In [20]:
# Create a copy
clean_category = category_df.copy()

# Convert month to datetime
clean_category["month"] = pd.to_datetime(
    clean_category["month"], errors="coerce"
)

# Remove rows with invalid month or missing category
clean_category = clean_category.dropna(
    subset=["month", "category"]
)

# Convert net inflow to numeric
clean_category["net_inflow_crore"] = pd.to_numeric(
    clean_category["net_inflow_crore"],
    errors="coerce"
)

# Remove duplicate records
clean_category = clean_category.drop_duplicates()

# Save cleaned dataset
clean_category.to_csv(
    PROCESSED_DIR / "clean_category_inflows.csv",
    index=False
)

# Verify
print("Original Shape:", category_df.shape)
print("Cleaned Shape:", clean_category.shape)
print("\nMissing Values:")
print(clean_category.isnull().sum())
print("\nDuplicate Rows:", clean_category.duplicated().sum())

print("\nCategory Inflows Cleaning Completed!")

Original Shape: (144, 3)
Cleaned Shape: (144, 3)

Missing Values:
month               0
category            0
net_inflow_crore    0
dtype: int64

Duplicate Rows: 0

Category Inflows Cleaning Completed!


In [21]:
# Load Industry Folio Count dataset
folio_df = pd.read_csv(
    RAW_DIR / "06_industry_folio_count.csv"
)

print("Shape:", folio_df.shape)
print("\nColumns:")
print(folio_df.columns.tolist())

print("\nData Types:")
print(folio_df.dtypes)

print("\nFirst 5 Rows:")
display(folio_df.head())

print("\nMissing Values:")
print(folio_df.isnull().sum())

print("\nDuplicate Rows:")
print(folio_df.duplicated().sum())

Shape: (21, 6)

Columns:
['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']

Data Types:
month                   object
total_folios_crore     float64
equity_folios_crore    float64
debt_folios_crore      float64
hybrid_folios_crore    float64
others_folios_crore    float64
dtype: object

First 5 Rows:


,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01,13.26,9.28,1.86,0.80,1.33
1,2022-04,13.91,9.74,1.95,0.83,1.39
2,2022-07,13.85,9.69,1.94,0.83,1.38
3,2022-10,14.12,9.88,1.98,0.85,1.41
4,2023-01,14.81,10.37,2.07,0.89,1.48



Missing Values:
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64

Duplicate Rows:
0


In [22]:
# Create a copy
clean_folio = folio_df.copy()

# Convert month to datetime
clean_folio["month"] = pd.to_datetime(
    clean_folio["month"], errors="coerce"
)

# Convert numeric columns
numeric_columns = [
    "total_folios_crore",
    "equity_folios_crore",
    "debt_folios_crore",
    "hybrid_folios_crore",
    "others_folios_crore"
]

for col in numeric_columns:
    clean_folio[col] = pd.to_numeric(
        clean_folio[col], errors="coerce"
    )

# Remove invalid dates
clean_folio = clean_folio.dropna(subset=["month"])

# Remove duplicate records
clean_folio = clean_folio.drop_duplicates()

# Save cleaned dataset
clean_folio.to_csv(
    PROCESSED_DIR / "clean_industry_folio_count.csv",
    index=False
)

# Verify
print("Original Shape:", folio_df.shape)
print("Cleaned Shape:", clean_folio.shape)

print("\nMissing Values:")
print(clean_folio.isnull().sum())

print("\nDuplicate Rows:", clean_folio.duplicated().sum())

print("\nIndustry Folio Count Cleaning Completed!")

Original Shape: (21, 6)
Cleaned Shape: (21, 6)

Missing Values:
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64

Duplicate Rows: 0

Industry Folio Count Cleaning Completed!


In [23]:
holdings_df = pd.read_csv(
    RAW_DIR / "09_portfolio_holdings.csv"
)

print("Shape:", holdings_df.shape)
print("\nColumns:")
print(holdings_df.columns.tolist())

print("\nData Types:")
print(holdings_df.dtypes)

print("\nFirst 5 Rows:")
display(holdings_df.head())

print("\nMissing Values:")
print(holdings_df.isnull().sum())

print("\nDuplicate Rows:")
print(holdings_df.duplicated().sum())

Shape: (322, 8)

Columns:
['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct', 'market_value_cr', 'current_price_inr', 'portfolio_date']

Data Types:
amfi_code              int64
stock_symbol          object
stock_name            object
sector                object
weight_pct           float64
market_value_cr      float64
current_price_inr    float64
portfolio_date        object
dtype: object

First 5 Rows:


,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31



Missing Values:
amfi_code            0
stock_symbol         0
stock_name           0
sector               0
weight_pct           0
market_value_cr      0
current_price_inr    0
portfolio_date       0
dtype: int64

Duplicate Rows:
0


In [24]:
# Create a copy
clean_holdings = holdings_df.copy()

# Remove extra spaces from text columns
text_columns = clean_holdings.select_dtypes(
    include="object"
).columns

for col in text_columns:
    clean_holdings[col] = clean_holdings[col].str.strip()

# Convert numeric columns
numeric_columns = [
    "weight_pct",
    "market_value_cr",
    "current_price_inr"
]

for col in numeric_columns:
    clean_holdings[col] = pd.to_numeric(
        clean_holdings[col], errors="coerce"
    )

# Convert portfolio date
clean_holdings["portfolio_date"] = pd.to_datetime(
    clean_holdings["portfolio_date"], errors="coerce"
)

# Remove invalid records
clean_holdings = clean_holdings.dropna(
    subset=["amfi_code", "stock_symbol", "portfolio_date"]
)

# Remove duplicate records
clean_holdings = clean_holdings.drop_duplicates()

# Save cleaned dataset
clean_holdings.to_csv(
    PROCESSED_DIR / "clean_portfolio_holdings.csv",
    index=False
)

# Verify
print("Original Shape:", holdings_df.shape)
print("Cleaned Shape:", clean_holdings.shape)

print("\nMissing Values:")
print(clean_holdings.isnull().sum())

print("\nDuplicate Rows:", clean_holdings.duplicated().sum())

print("\nPortfolio Holdings Cleaning Completed!")

Original Shape: (322, 8)
Cleaned Shape: (322, 8)

Missing Values:
amfi_code            0
stock_symbol         0
stock_name           0
sector               0
weight_pct           0
market_value_cr      0
current_price_inr    0
portfolio_date       0
dtype: int64

Duplicate Rows: 0

Portfolio Holdings Cleaning Completed!


In [25]:
# Load Benchmark Indices dataset
benchmark_df = pd.read_csv(
    RAW_DIR / "10_benchmark_indices.csv"
)

print("Shape:", benchmark_df.shape)

print("\nColumns:")
print(benchmark_df.columns.tolist())

print("\nData Types:")
print(benchmark_df.dtypes)

print("\nFirst 5 Rows:")
display(benchmark_df.head())

print("\nMissing Values:")
print(benchmark_df.isnull().sum())

print("\nDuplicate Rows:")
print(benchmark_df.duplicated().sum())

Shape: (8050, 3)

Columns:
['date', 'index_name', 'close_value']

Data Types:
date            object
index_name      object
close_value    float64
dtype: object

First 5 Rows:


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15



Missing Values:
date           0
index_name     0
close_value    0
dtype: int64

Duplicate Rows:
0


In [26]:
# Create a copy
clean_benchmark = benchmark_df.copy()

# Convert date to datetime
clean_benchmark["date"] = pd.to_datetime(
    clean_benchmark["date"], errors="coerce"
)

# Clean index names
clean_benchmark["index_name"] = (
    clean_benchmark["index_name"].str.strip()
)

# Convert closing value to numeric
clean_benchmark["close_value"] = pd.to_numeric(
    clean_benchmark["close_value"], errors="coerce"
)

# Remove invalid records
clean_benchmark = clean_benchmark.dropna(
    subset=["date", "index_name", "close_value"]
)

# Remove duplicate records
clean_benchmark = clean_benchmark.drop_duplicates()

# Sort records
clean_benchmark = clean_benchmark.sort_values(
    by=["index_name", "date"]
)

# Save cleaned dataset
clean_benchmark.to_csv(
    PROCESSED_DIR / "clean_benchmark_indices.csv",
    index=False
)

# Verify
print("Original Shape:", benchmark_df.shape)
print("Cleaned Shape:", clean_benchmark.shape)

print("\nMissing Values:")
print(clean_benchmark.isnull().sum())

print("\nDuplicate Rows:", clean_benchmark.duplicated().sum())

print("\nBenchmark Indices Cleaning Completed!")

Original Shape: (8050, 3)
Cleaned Shape: (8050, 3)

Missing Values:
date           0
index_name     0
close_value    0
dtype: int64

Duplicate Rows: 0

Benchmark Indices Cleaning Completed!


#### Connect to the database

In [27]:
from sqlalchemy import create_engine, inspect

# Connect to SQLite database
DB_PATH = PROJECT_ROOT / "data" / "db" / "bluestock_mf.db"

engine = create_engine(f"sqlite:///{DB_PATH}")

print("Database Connected Successfully!")

Database Connected Successfully!


In [28]:
# Load cleaned Fund Master dataset
fund_df = pd.read_csv(PROCESSED_DIR / "clean_fund_master.csv")

# Select columns matching dim_fund table
fund_df = fund_df[
    ["amfi_code", "scheme_name", "fund_house",
     "category", "sub_category", "plan"]
]

# Insert data into database
fund_df.to_sql(
    "dim_fund",
    engine,
    if_exists="append",
    index=False
)

print("Fund Master loaded successfully!")

Fund Master loaded successfully!


In [29]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM dim_fund"
    ).scalar()

print("Total Funds Loaded:", count)

Total Funds Loaded: 40


In [31]:
# Load cleaned NAV dataset
nav_df = pd.read_csv(PROCESSED_DIR / "clean_nav.csv")

# Rename date column to match database schema
nav_df = nav_df.rename(columns={"date": "nav_date"})

# Insert into database
nav_df.to_sql(
    "fact_nav",
    engine,
    if_exists="append",
    index=False
)

print("NAV History loaded successfully!")

NAV History loaded successfully!


In [ ]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM fact_nav"
    ).scalar()

print("Total NAV Records Loaded:", count)

Total NAV Records Loaded: 46000


In [33]:
# Load cleaned transactions dataset
transactions_df = pd.read_csv(
    PROCESSED_DIR / "clean_transactions.csv"
)

# Insert into database
transactions_df.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

print("Investor Transactions loaded successfully!")

Investor Transactions loaded successfully!


In [34]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM fact_transactions"
    ).scalar()

print("Total Transactions Loaded:", count)

Total Transactions Loaded: 32778


In [35]:
# Load cleaned Scheme Performance dataset
performance_df = pd.read_csv(
    PROCESSED_DIR / "clean_performance.csv"
)

# Select columns matching fact_performance table
performance_df = performance_df[
    [
        "amfi_code",
        "return_1yr_pct",
        "return_3yr_pct",
        "return_5yr_pct",
        "benchmark_3yr_pct",
        "alpha",
        "beta",
        "sharpe_ratio",
        "sortino_ratio",
        "std_dev_ann_pct",
        "max_drawdown_pct",
        "aum_crore",
        "expense_ratio_pct",
        "morningstar_rating",
        "risk_grade"
    ]
]

# Insert into database
performance_df.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

print("Scheme Performance loaded successfully!")

Scheme Performance loaded successfully!


In [36]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM fact_performance"
    ).scalar()

print("Total Performance Records Loaded:", count)

Total Performance Records Loaded: 40


In [39]:
with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS fact_aum")

    conn.exec_driver_sql("""
        CREATE TABLE fact_aum (
            aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
            fund_house TEXT NOT NULL,
            report_year INTEGER NOT NULL,
            aum_crore REAL NOT NULL
        )
    """)

print("AUM table updated successfully!")

AUM table updated successfully!


In [40]:
aum_df = pd.read_csv(PROCESSED_DIR / "clean_aum.csv")

aum_df["date"] = pd.to_datetime(aum_df["date"])
aum_df["report_year"] = aum_df["date"].dt.year

aum_df = aum_df[
    ["fund_house", "report_year", "aum_crore"]
]

aum_df.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

print("AUM Data loaded successfully!")

AUM Data loaded successfully!


In [41]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM fact_aum"
    ).scalar()

print("Total AUM Records Loaded:", count)

Total AUM Records Loaded: 90


In [42]:
# Load cleaned Monthly SIP dataset
sip_df = pd.read_csv(
    PROCESSED_DIR / "clean_monthly_sip.csv"
)

# Insert into SQLite database
sip_df.to_sql(
    "monthly_sip_inflows",
    engine,
    if_exists="append",
    index=False
)

print("Monthly SIP data loaded successfully!")

Monthly SIP data loaded successfully!


In [43]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM monthly_sip_inflows"
    ).scalar()

print("Total SIP Records Loaded:", count)

Total SIP Records Loaded: 48


In [44]:
# Load cleaned Category Inflows dataset
category_df = pd.read_csv(
    PROCESSED_DIR / "clean_category_inflows.csv"
)

# Insert into SQLite database
category_df.to_sql(
    "category_inflows",
    engine,
    if_exists="append",
    index=False
)

print("Category Inflows loaded successfully!")

Category Inflows loaded successfully!


In [45]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM category_inflows"
    ).scalar()

print("Total Category Records Loaded:", count)

Total Category Records Loaded: 144


In [46]:
# Load cleaned Industry Folio Count dataset
folio_df = pd.read_csv(
    PROCESSED_DIR / "clean_industry_folio_count.csv"
)

# Insert into SQLite database
folio_df.to_sql(
    "industry_folio_count",
    engine,
    if_exists="append",
    index=False
)

print("Industry Folio Count loaded successfully!")

Industry Folio Count loaded successfully!


In [47]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM industry_folio_count"
    ).scalar()

print("Total Folio Records Loaded:", count)

Total Folio Records Loaded: 21


In [48]:
# Load cleaned Portfolio Holdings dataset
holdings_df = pd.read_csv(
    PROCESSED_DIR / "clean_portfolio_holdings.csv"
)

# Insert into SQLite database
holdings_df.to_sql(
    "portfolio_holdings",
    engine,
    if_exists="append",
    index=False
)

print("Portfolio Holdings loaded successfully!")

Portfolio Holdings loaded successfully!


Portfolio Holdings loaded successfully!


In [50]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM portfolio_holdings"
    ).scalar()

print("Total Portfolio Records Loaded:", count)

Total Portfolio Records Loaded: 644


In [51]:
# Load cleaned Benchmark Indices dataset
benchmark_df = pd.read_csv(
    PROCESSED_DIR / "clean_benchmark_indices.csv"
)

# Insert into SQLite database
benchmark_df.to_sql(
    "benchmark_indices",
    engine,
    if_exists="append",
    index=False
)

print("Benchmark Indices loaded successfully!")

Benchmark Indices loaded successfully!


In [52]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM benchmark_indices"
    ).scalar()

print("Total Benchmark Records Loaded:", count)

Total Benchmark Records Loaded: 8050


In [53]:
# Verify all database tables
inspector = inspect(engine)

tables = inspector.get_table_names()

print("Database Tables and Record Counts:\n")

for table in tables:
    with engine.connect() as conn:
        count = conn.exec_driver_sql(
            f"SELECT COUNT(*) FROM {table}"
        ).scalar()

    print(f"{table}: {count} records")

Database Tables and Record Counts:

benchmark_indices: 8050 records
category_inflows: 144 records
dim_date: 0 records
dim_fund: 40 records
fact_aum: 90 records
fact_nav: 46000 records
fact_performance: 40 records
fact_transactions: 32778 records
industry_folio_count: 21 records
monthly_sip_inflows: 48 records
portfolio_holdings: 644 records


In [54]:
# Collect dates from cleaned datasets
date_columns = {
    "clean_nav.csv": "date",
    "clean_transactions.csv": "transaction_date",
    "clean_aum.csv": "date",
    "clean_monthly_sip.csv": "month",
    "clean_category_inflows.csv": "month",
    "clean_industry_folio_count.csv": "month",
    "clean_portfolio_holdings.csv": "portfolio_date",
    "clean_benchmark_indices.csv": "date"
}

all_dates = []

for file_name, date_col in date_columns.items():
    df = pd.read_csv(PROCESSED_DIR / file_name)
    dates = pd.to_datetime(df[date_col], errors="coerce")
    all_dates.extend(dates.dropna().dt.normalize().tolist())

# Get unique dates
unique_dates = sorted(set(all_dates))

# Create date dimension DataFrame
dim_date_df = pd.DataFrame({
    "full_date": unique_dates
})

dim_date_df["date_id"] = range(1, len(dim_date_df) + 1)
dim_date_df["day"] = dim_date_df["full_date"].dt.day
dim_date_df["month"] = dim_date_df["full_date"].dt.month
dim_date_df["month_name"] = dim_date_df["full_date"].dt.month_name()
dim_date_df["quarter"] = dim_date_df["full_date"].dt.quarter
dim_date_df["year"] = dim_date_df["full_date"].dt.year

# Insert into database
dim_date_df.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

print("Date Dimension Loaded Successfully!")
print("Total Dates:", len(dim_date_df))

Date Dimension Loaded Successfully!
Total Dates: 1306


In [55]:
with engine.connect() as conn:
    count = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM dim_date"
    ).scalar()

print("Total Dates in dim_date:", count)

Total Dates in dim_date: 1306


### SQL QUERY

In [57]:
# Read SQL query from file
with open(PROJECT_ROOT / "sql" / "queries.sql", "r") as file:
    query = file.read()

# Execute query
result = pd.read_sql(query, engine)

# Display result
display(result)

,scheme_name,fund_house,aum_crore
0,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
1,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
2,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
4,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0


In [58]:
query = """
SELECT
    strftime('%Y-%m', nav_date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY strftime('%Y-%m', nav_date)
ORDER BY month;
"""

result = pd.read_sql_query(query, engine)
display(result)

,month,average_nav
0,2022-01,207.06
1,2022-02,207.72
2,2022-03,209.69
3,2022-04,211.83
4,2022-05,212.73
5,2022-06,213.86
6,2022-07,213.96
7,2022-08,215.68
8,2022-09,218.49
9,2022-10,219.53


In [59]:
query = """
SELECT
    month,
    sip_inflow_crore,
    yoy_growth_pct
FROM monthly_sip_inflows
WHERE yoy_growth_pct IS NOT NULL
ORDER BY month;
"""

result = pd.read_sql_query(query, engine)
display(result)

,month,sip_inflow_crore,yoy_growth_pct
0,2023-01-01,13856,20.31
1,2023-02-01,13687,19.66
2,2023-03-01,14276,15.80
3,2023-04-01,14749,24.33
4,2023-05-01,14749,20.05
5,2023-06-01,14734,20.02
6,2023-07-01,15245,25.58
7,2023-08-01,15814,24.58
8,2023-09-01,16042,23.63
9,2023-10-01,16928,29.82


In [63]:
sql_file = PROJECT_ROOT / "sql" / "queries.sql"

print("File exists:", sql_file.exists())

with open(sql_file, "r") as file:
    sql_content = file.read()

print(sql_content)

File exists: True


-- Question: Find the top five mutual funds
-- with the highest Assets Under Management (AUM).

-- Query 1: Top 5 Funds by AUM

SELECT
    f.scheme_name,
    a.fund_house,
    a.aum_crore
FROM fact_aum a
JOIN dim_fund f
    ON a.fund_house = f.fund_house
ORDER BY a.aum_crore DESC
LIMIT 5;


-- Query 2: Average NAV per Month
SELECT
    strftime('%Y-%m', nav_date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY strftime('%Y-%m', nav_date)
ORDER BY month;


-- Query 3: SIP Inflow YoY Growth
SELECT
    month,
    sip_inflow_crore,
    yoy_growth_pct
FROM monthly_sip_inflows
WHERE yoy_growth_pct IS NOT NULL
ORDER BY month;


-- Query 4: Transactions by State
SELECT state,
       COUNT(*) AS total_transactions,
       ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC;


-- Query 5: Funds with Expense Ratio Below 1%
SELECT f.scheme_name,
       f.fund_house,
       p.expense_ratio
FROM di

### execution of all queries 

In [65]:
import re
import pandas as pd

queries = re.findall(
    r'(?is)(SELECT\b.*?);',
    sql_content
)

print("Total queries:", len(queries))

for i, query in enumerate(queries, start=1):
    print(f"\n========== Query {i} ==========")
    try:
        result = pd.read_sql_query(query, engine)
        display(result)
    except Exception as e:
        print("Error:", e)

Total queries: 15

========== Query 1 ==========


,scheme_name,fund_house,aum_crore
0,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
1,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
2,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
4,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0



========== Query 2 ==========


,month,average_nav
0,2022-01,207.06
1,2022-02,207.72
2,2022-03,209.69
3,2022-04,211.83
4,2022-05,212.73
5,2022-06,213.86
6,2022-07,213.96
7,2022-08,215.68
8,2022-09,218.49
9,2022-10,219.53



========== Query 3 ==========


,month,sip_inflow_crore,yoy_growth_pct
0,2023-01-01,13856,20.31
1,2023-02-01,13687,19.66
2,2023-03-01,14276,15.80
3,2023-04-01,14749,24.33
4,2023-05-01,14749,20.05
5,2023-06-01,14734,20.02
6,2023-07-01,15245,25.58
7,2023-08-01,15814,24.58
8,2023-09-01,16042,23.63
9,2023-10-01,16928,29.82


... (14 outputs left out)


========== Query 13 ==========
Error: (sqlite3.OperationalError) no such column: p.return_1y
[SQL: SELECT
    f.category,
    ROUND(AVG(p.return_1y), 2) AS average_1y_return
FROM dim_fund f
JOIN fact_performance p
    ON f.amfi_code = p.amfi_code
GROUP BY f.category
ORDER BY average_1y_return DESC]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

========== Query 14 ==========


,index_name,month,average_close_value
0,BSE_SMALLCAP,2022-01,26646.45
1,CRISIL_GILT,2022-01,1452.45
2,CRISIL_LIQUID,2022-01,2286.36
3,NIFTY100,2022-01,17939.59
4,NIFTY50,2022-01,18167.48
...,...,...,...
366,CRISIL_LIQUID,2026-05,3035.75
367,NIFTY100,2026-05,19052.77
368,NIFTY50,2026-05,19078.12
369,NIFTY500,2026-05,37065.19



========== Query 15 ==========


,sector,total_weight_pct
0,Banking,1304.52
1,IT,910.94
2,Pharma,814.90
3,Automobile,647.30
4,Utilities,531.08
5,FMCG,458.22
6,Infrastructure,384.32
7,Diversified,338.46
8,Telecom,291.24
9,Consumer Goods,255.22
